# AIRPATH-AI Milestone 3C — target-time PM2.5 integration

This notebook connects the frozen station forecaster, IDW p=1 spatial estimator, and ordered road-segment ETAs.

Safeguards:

- non-hourly ETAs are explicitly ceiled to the next supported hour without interpolation;
- only frozen t+1h/t+2h/t+3h forecasts are accepted;
- deployment mode accepts exact pre-origin lag bundles, never future observations;
- oracle observations are kept in a separate pathway;
- outputs are segment PM2.5 estimates, not exposure or route recommendations;
- second-level ETAs do not imply minute-level PM2.5 accuracy.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.target_time_integration import (
    EXAMPLE_FORECASTING_ORIGIN,
    generate_integration_outputs,
    map_target_time,
)

In [ ]:
mapping_examples = [
    map_target_time("2022-02-28 07:00:00", EXAMPLE_FORECASTING_ORIGIN),
    map_target_time("2022-02-28 06:03:00", EXAMPLE_FORECASTING_ORIGIN),
    map_target_time("2022-02-28 09:01:00", EXAMPLE_FORECASTING_ORIGIN),
]
pd.DataFrame(
    {
        "requested_target_time": [item.requested_target_time for item in mapping_examples],
        "supported_target_time": [item.supported_target_time for item in mapping_examples],
        "mapping_method": [item.mapping_method for item in mapping_examples],
        "horizon_hours": [item.forecast_horizon_hours for item in mapping_examples],
        "supported": [item.supported for item in mapping_examples],
        "status": [item.status for item in mapping_examples],
    }
)

In [ ]:
outputs = generate_integration_outputs(
    network_path=PROJECT_ROOT / "data/processed/road_network/healthyair_pilot_osm.json.gz",
    clean_csv=PROJECT_ROOT / "data/processed/airquality_hcmc_clean.csv",
    forecaster_path=PROJECT_ROOT / "data/processed/models/hourly_station_forecaster.joblib",
    processed_directory=PROJECT_ROOT / "data/processed/target_time",
    report_root=PROJECT_ROOT / "reports",
)
outputs["summary"]

In [ ]:
sample_columns = [
    "station_value_source",
    "mode",
    "route_id",
    "segment_index",
    "requested_target_time",
    "supported_target_time",
    "forecast_horizon_hours",
    "predicted_pm25",
    "reliability_status",
]
display(outputs["sample"][sample_columns])

In [ ]:
deployment_first = outputs["deployment_records"][0]
oracle_first = outputs["oracle_records"][0]
pd.DataFrame(
    {
        "pathway": [oracle_first.station_value_source, deployment_first.station_value_source],
        "station_values_target_time": [
            oracle_first.station_values_target_time,
            deployment_first.station_values_target_time,
        ],
        "station_values_used": [
            dict(oracle_first.station_values_used),
            dict(deployment_first.station_values_used),
        ],
        "segment_pm25": [oracle_first.predicted_pm25, deployment_first.predicted_pm25],
    }
)

## Interpretation boundary

The output is `PM2.5(segment midpoint, supported hourly target)`. For example, a 06:03 passage maps to 07:00 and records the 57-minute offset. It is not an interpolated 06:03 observation and does not establish minute-level predictive accuracy.

No segment values are summed or duration-weighted here. Exposure aggregation and route optimization remain outside Milestone 3C.